ttfb_s — time to first token

time_to_final_token_s — time from first token to last token

total_s — end-to-end (≈ TTFB + time_to_final_token)

It includes conditional thinking for 2.5-Pro, skips non-text models, retries 429/503, and exports a CSV.

#### Cell 1 — Install

In [ ]:
!pip install -U google-genai python-dotenv pandas tabulate


#### Cell 2 — Load API key

In [ ]:
from dotenv import load_dotenv
import os

# change "keys.env" to ".env" if that's your filename
load_dotenv("keys.env")

def masked(v): 
    return v[:4] + "..." + v[-4:] if v and len(v) >= 8 else str(bool(v))

print("GEMINI_API_KEY:", masked(os.getenv("GEMINI_API_KEY")))
assert os.getenv("GEMINI_API_KEY"), "Missing GEMINI_API_KEY in keys.env/.env"


#### Cell 3 — Imports & global config

In [ ]:
import time, math, statistics as stats, pandas as pd
from tabulate import tabulate
from typing import List, Tuple, Dict, Any
from google import genai
from google.genai import types

PROMPT = "Explain transformers in AI in 3 short sentences."
N_RUNS = 3                # repeats per model (excluding warm-up)
MAX_OUTPUT_TOKENS = 128   # keep answers short/consistent
TEMPERATURE = 0.5         # moderately deterministic

client = genai.Client()   # uses GEMINI_API_KEY from env


#### Cell 4 — Model helpers (filters & rules)

In [ ]:
def needs_thinking(model_id: str) -> bool:
    """Gemini 2.5 Pro family requires thinking mode."""
    return "gemini-2.5-pro" in model_id

def disallow_thinking(model_id: str) -> bool:
    """Gemma family does not support thinking_config."""
    return model_id.startswith("models/gemma-")

def is_text_model(model_id: str) -> bool:
    """Filter out non-text models (imagen/veo/embeddings/tts/live/image)."""
    bad = ["imagen", "veo", "embedding", "aqa", "tts", "live", "image"]
    return all(b not in model_id for b in bad)

def normalize_ids(ids):
    def norm(mid):
        return mid if mid.startswith("models/") else f"models/{mid}"
    return [norm(m) for m in ids]


#### Cell 5 — Runner (streaming + conditional thinking + retries)

In [ ]:
def run_once(model_id: str, prompt: str) -> Dict[str, Any]:
    """
    Streams the response and measures:
      - ttfb_s (time to first token)
      - time_to_final_token_s (time from first token to last token)
      - total_s (end-to-end)
    Applies:
      - thinking_config=256 for 2.5 Pro (valid range 128..32768)
      - thinking_config omitted for Gemma
      - thinking_config=0 for other Gemini
    Retries transient 429/503 errors.
    """
    retries = 2
    backoff = 1.5
    attempt = 0

    while True:
        try:
            t0 = time.perf_counter()
            first = None
            last = None
            chunks = []

            cfg = dict(
                temperature=TEMPERATURE,
                max_output_tokens=MAX_OUTPUT_TOKENS,
                candidate_count=1,
            )
            if needs_thinking(model_id):
                cfg["thinking_config"] = types.ThinkingConfig(thinking_budget=256)
            elif disallow_thinking(model_id):
                pass  # omit thinking_config
            else:
                cfg["thinking_config"] = types.ThinkingConfig(thinking_budget=0)

            stream = client.models.generate_content_stream(
                model=model_id,
                contents=prompt,
                config=types.GenerateContentConfig(**cfg),
            )

            for chunk in stream:
                if chunk.text:
                    now = time.perf_counter()
                    if first is None:
                        first = now
                    last = now
                    chunks.append(chunk.text)

            total_s = time.perf_counter() - t0
            ttfb_s = (first - t0) if first else math.nan
            time_to_final_token_s = (last - first) if (first and last) else math.nan

            return {
                "model": model_id,
                "ttfb_s": ttfb_s,
                "time_to_final_token_s": time_to_final_token_s,
                "total_s": total_s,
                "text": "".join(chunks),
            }

        except Exception as e:
            msg = str(e)
            transient = ("RESOURCE_EXHAUSTED" in msg) or ("UNAVAILABLE" in msg) or ("429" in msg) or ("503" in msg)
            if transient and attempt < retries:
                attempt += 1
                time.sleep(backoff * attempt)
                continue
            raise


#### Cell 6 — Benchmark function

In [ ]:
def bench_models(models: List[str], prompt: str) -> pd.DataFrame:
    rows = []
    for mid in models:
        print(f"\n--- {mid} ---")
        try:
            # Warm-up (ignore timing)
            _ = run_once(mid, prompt)

            # Measured runs
            times = [run_once(mid, prompt) for _ in range(N_RUNS)]

            ttfb_vals = [t["ttfb_s"] for t in times if not math.isnan(t["ttfb_s"])]
            t2last_vals = [t["time_to_final_token_s"] for t in times if not math.isnan(t["time_to_final_token_s"])]
            total_vals = [t["total_s"] for t in times]

            def summary(xs: List[float]) -> Tuple[float,float,float]:
                return (min(xs), sum(xs)/len(xs), max(xs))

            ttfb = summary(ttfb_vals) if ttfb_vals else (math.nan, math.nan, math.nan)
            t2last = summary(t2last_vals) if t2last_vals else (math.nan, math.nan, math.nan)
            total = summary(total_vals)

            print(f"TTFB               min/avg/max: {ttfb}")
            print(f"Time-to-final      min/avg/max: {t2last}")
            print(f"Total              min/avg/max: {total}")

            rows.append({
                "model": mid,
                "ttfb_min": ttfb[0], "ttfb_avg": ttfb[1], "ttfb_max": ttfb[2],
                "t2last_min": t2last[0], "t2last_avg": t2last[1], "t2last_max": t2last[2],
                "total_min": total[0], "total_avg": total[1], "total_max": total[2],
            })

        except Exception as e:
            print(f"Error: {e}")
            rows.append({
                "model": mid,
                "ttfb_min": None, "ttfb_avg": None, "ttfb_max": None,
                "t2last_min": None, "t2last_avg": None, "t2last_max": None,
                "total_min": None, "total_avg": None, "total_max": None,
            })

    df = pd.DataFrame(
        rows,
        columns=[
            "model",
            "ttfb_min","ttfb_avg","ttfb_max",
            "t2last_min","t2last_avg","t2last_max",
            "total_min","total_avg","total_max",
        ]
    )

    if not df.empty:
        df = df.sort_values("total_avg", na_position="last")
    else:
        print("No successful rows collected (empty DataFrame).")
    return df


#### Cell 7 — Choose models to test (from your list)

In [ ]:
# Pick a representative set (you can edit as needed)
wanted = normalize_ids([
    # Gemini 1.5 (older; often free-tier)
    "gemini-1.5-flash",
    "gemini-1.5-pro",

    # Gemini 2.0 (legacy/experimental speed tiers)
    "gemini-2.0-flash",
    "gemini-2.0-flash-lite",

    # Gemini 2.5 (current mainline; best for production)
    "gemini-2.5-flash",
    "gemini-2.5-flash-lite",
    "gemini-2.5-pro",

    # Optional: open-weight via API (omit if not needed)
    "gemma-3-4b-it",
])

available = {m.name for m in client.models.list()}
to_test = [m for m in wanted if (m in available and is_text_model(m))]

print("Testing these models:", to_test)
assert len(to_test) > 0, "After filtering, no models remain. Check model IDs or permissions."


#### Cell 8 — Run & export

In [ ]:
df = bench_models(to_test, PROMPT)

print("\n=== Sorted by total_avg (lower is faster) ===")
if not df.empty:
    print(tabulate(df, headers="keys", tablefmt="github", floatfmt=".3f"))
    df.to_csv("gemini_latency_comparison.csv", index=False)
    print("\nSaved: gemini_latency_comparison.csv")
else:
    print("No results to show.")
